# 04 Statistical Analysis

Use this notebook for deeper analysis such as correlation checks, hypothesis testing, forecasting, segmentation, or regression.

In [ ]:
from pathlib import Path

import pandas as pd
from scipy import stats

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data/processed/tableau_ready_dataset.csv'
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Correlation Analysis
How strongly does starting position (grid) relate to finishing position?

In [ ]:
correlation = df[['grid', 'positionOrder']].corr(method='spearman')
print(f"Spearman Correlation between Grid and Position Order: {correlation.iloc[0, 1]:.3f}")

**Business Interpretation:** A strong positive correlation indicates that qualifying is critical for race success.

## 2. Hypothesis Testing: Top 5 Grid Advantage
Do drivers starting in the Top 5 score significantly more points?

In [ ]:
top_5 = df[df['grid'] <= 5]['points']
rest = df[df['grid'] > 5]['points']

t_stat, p_val = stats.ttest_ind(top_5, rest, equal_var=False)
print(f"T-Statistic: {t_stat:.3f}")
print(f"P-Value: {p_val:.3e}")

**Business Interpretation:** If P < 0.05, we reject the null hypothesis and conclude that Top 5 starters have a statistically significant advantage in scoring points.

## 3. Chi-Square Test: Pole Position vs. Race Win
Is winning independent of starting on pole?

In [ ]:
contingency_table = pd.crosstab(df['grid'] == 1, df['is_win'] == 1)
chi2, p, dof, ex = stats.chi2_contingency(contingency_table)
print(f"Chi-Square Stat: {chi2:.3f}")
print(f"P-Value: {p:.3e}")

**Business Interpretation:** A low P-value suggests that pole position and race wins are highly dependent.

## 4. Regression Analysis: Predicting Position Order
Quantifying the impact of grid position on finishing position.

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(df['grid'])
y = df['positionOrder']
model = sm.OLS(y, X).fit()
print(model.summary())

**Business Interpretation:** The coefficient for 'grid' tells us how many positions a driver moves back (on average) for each spot lower on the grid.

## 5. DNF Analysis: Reliability vs. Performance
Are certain teams more prone to mechanical failures?

In [ ]:
# Filter for non-finishing statuses (exclude 'Finished' and '+n Laps')
dnf_df = df[~df['status'].isin(['Finished', '+1 Lap', '+2 Laps', '+3 Laps', '+4 Laps', '+5 Laps'])]
top_dnf_teams = dnf_df.groupby('team_name').size().sort_values(ascending=False).head(10)
print("Top 10 Teams by DNF Count:")
print(top_dnf_teams)

**Business Interpretation:** High DNF counts for top teams indicate reliability issues that could be investigated further (e.g., engine vs. accidental).

## 6. Advanced Regression: Predicting Points
Using grid position and constructor to predict points scored.

In [ ]:
# Create dummy variables for top constructors to see their impact
top_teams = df.groupby('team_name')['points'].sum().nlargest(5).index
df['is_top_team'] = df['team_name'].isin(top_teams).astype(int)

X_adv = sm.add_constant(df[['grid', 'is_top_team']])
y_adv = df['points']
model_adv = sm.OLS(y_adv, X_adv).fit()
print(model_adv.summary())

**Business Interpretation:** This model quantifies the 'points premium' of being in a top-tier constructor (is_top_team) versus starting position (grid).